# GigLedger — OCR Notebook
**Phase 2 · Step 1** — Interactive visualisation of the OCR pipeline.

All core logic lives in `../ocr_engine.py` (the production module).
This notebook imports from it directly — so notebook and FastAPI service
always use exactly the same extraction code.

---
## Environment check
All dependencies should already be installed. If not:
```bash
pip install --user pytesseract pillow opencv-python-headless matplotlib numpy jupyter
```
Tesseract binary path is auto-configured in `ocr_engine.py`.

In [ ]:
# ─────────────────────────────────────────────
#  CELL 1 — Imports & environment check
# ─────────────────────────────────────────────
import sys
import os
from pathlib import Path
import json

import cv2
import numpy as np
import matplotlib.pyplot as plt
import pytesseract

# Add the ml-service root to path so we can import the production module
ML_ROOT = Path(os.getcwd()).parent
if str(ML_ROOT) not in sys.path:
    sys.path.insert(0, str(ML_ROOT))

# Import production OCR engine
from ocr_engine import (
    preprocess_image,
    run_ocr,
    extract_fields,
    compute_confidence,
    process_image_path,
)

# Verify Tesseract
try:
    version = pytesseract.get_tesseract_version()
    print(f"Tesseract OK — version {version}")
except Exception as e:
    print(f"Tesseract ERROR: {e}")

SAMPLE_DIR  = ML_ROOT / 'sample_images'
RESULTS_DIR = ML_ROOT / 'ocr_results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Sample images dir: {SAMPLE_DIR}")
images_found = list(SAMPLE_DIR.glob('*.png')) + list(SAMPLE_DIR.glob('*.jpg'))
print(f"Images available:  {len(images_found)}")
for p in images_found:
    print(f"  {p.name}")

## Cell 2 — Generate test images (or point to your own)

**Option A** — Run the cell below to generate synthetic test images automatically.

**Option B** — Copy your real Zomato/Swiggy screenshots into `ml-service/sample_images/`
and then set `IMAGE_PATHS` manually.

In [ ]:
# ─────────────────────────────────────────────
#  CELL 2 — Generate synthetic test images
# ─────────────────────────────────────────────

# Import generator from the notebooks folder
NOTEBOOKS_DIR = Path(os.getcwd())
sys.path.insert(0, str(NOTEBOOKS_DIR))
from create_test_images import TEST_CASES, make_zomato_card, make_swiggy_card, SAMPLE_DIR as SD

print('Generating synthetic screenshots...')
for tc in TEST_CASES:
    if tc['app'] == 'zomato':
        make_zomato_card(tc['filename'], tc['fare'], tc['distance'])
    else:
        make_swiggy_card(tc['filename'], tc['fare'], tc['distance'])
    print(f"  {tc['filename']}  (Rs.{tc['fare']} / {tc['distance']} km)")

# ── Set IMAGE_PATHS ──
# Leave as-is to use generated images, or replace with your own paths:
# IMAGE_PATHS = [
#     r'C:\Users\vimal\Pictures\zomato_real.png',
#     r'C:\Users\vimal\Pictures\swiggy_real.jpg',
# ]
IMAGE_PATHS = [str(SAMPLE_DIR / tc['filename']) for tc in TEST_CASES]
PRIMARY_IMAGE_PATH = IMAGE_PATHS[0]
print(f'\nUsing {len(IMAGE_PATHS)} images. Primary: {os.path.basename(PRIMARY_IMAGE_PATH)}')

## Cell 3 — Preprocessing Pipeline (inline visualisation)

Four stages displayed side-by-side for the primary image.
Check this to understand what Tesseract is actually seeing.

In [ ]:
# ─────────────────────────────────────────────
#  CELL 3 — Preprocessing visualisation
# ─────────────────────────────────────────────
raw_img = cv2.imread(PRIMARY_IMAGE_PATH)
stages  = preprocess_image(raw_img)

stage_keys   = ['original', 'grayscale', 'contrast_adjusted', 'thresholded']
stage_labels = ['Original', 'Grayscale', 'Contrast (CLAHE)', 'Otsu Threshold']

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle(f'Preprocessing Pipeline — {os.path.basename(PRIMARY_IMAGE_PATH)}',
             fontsize=13, fontweight='bold')

for ax, key, label in zip(axes, stage_keys, stage_labels):
    img = stages[key]
    if img.ndim == 2:
        ax.imshow(img, cmap='gray')
    else:
        ax.imshow(img)
    ax.set_title(label, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.show()
print('Preprocessing complete — check each stage above.')

## Cell 4 — Raw OCR Text

In [ ]:
# ─────────────────────────────────────────────
#  CELL 4 — Tesseract OCR output
# ─────────────────────────────────────────────
raw_text = run_ocr(stages['final'])

print('=' * 60)
print('RAW TESSERACT OUTPUT')
print('=' * 60)
print(raw_text)
print('=' * 60)
print(f'Characters extracted: {len(raw_text)}')

## Cell 5 — Regex Extraction + Confidence

In [ ]:
# ─────────────────────────────────────────────
#  CELL 5 — Extract fields & score confidence
# ─────────────────────────────────────────────
fields     = extract_fields(raw_text)
confidence = compute_confidence(fields, raw_text)

if confidence >= 0.75:
    level = 'HIGH'
elif confidence >= 0.45:
    level = 'MEDIUM — user should verify'
else:
    level = 'LOW — extraction unreliable'

print('Currency matches:', fields['currency_matches'])
print('Distance matches:', fields['distance_matches'])
print('Used fallback:   ', fields['used_fallback'])
print()
print('Structured result:')
print(json.dumps({
    'promised_amount': fields['promised_amount'],
    'distance_km':     fields['distance_km'],
    'confidence':      confidence,
    'confidence_level': level,
    'raw_text':        raw_text.strip()[:200]
}, indent=2, ensure_ascii=False))

## Cell 6 — Batch Test: All Images vs. Ground Truth

Fill in `EXPECTED` with the values you know from your screenshots.
Leave empty to run without accuracy checking.

In [ ]:
# ─────────────────────────────────────────────
#  CELL 6 — Batch test table
# ─────────────────────────────────────────────

# Fill in your real expected values here:
EXPECTED = [
    {'path': IMAGE_PATHS[0], 'amount': 85.0,  'distance': 3.5},
    {'path': IMAGE_PATHS[1], 'amount': 120.0, 'distance': 5.2},
    {'path': IMAGE_PATHS[2], 'amount': 65.0,  'distance': 2.1},
    # Add more rows here for your real screenshots:
    # {'path': r'C:\path\to\real_screenshot.png', 'amount': 95.0, 'distance': 4.1},
]

rows = []
for entry in EXPECTED:
    result  = process_image_path(entry['path'])
    exp_amt = entry.get('amount')
    exp_dst = entry.get('distance')
    got_amt = result['promised_amount']
    got_dst = result['distance_km']

    amt_ok  = (exp_amt is None) or (got_amt is not None and abs(got_amt - exp_amt) <= 1.0)
    dst_ok  = (exp_dst is None) or (got_dst is not None and abs(got_dst - exp_dst) <= 0.1)

    rows.append({
        'Image':    os.path.basename(entry['path']),
        'Exp Rs.':  exp_amt, 'Got Rs.':  got_amt, 'Fare OK':  'PASS' if amt_ok else 'FAIL',
        'Exp km':   exp_dst, 'Got km':   got_dst, 'Dist OK':  'PASS' if dst_ok else 'FAIL',
        'Conf':     result['confidence'],
    })

# Print table
print('=' * 95)
print(f"{'Image':<28} {'Exp Rs.':>8} {'Got Rs.':>8} {'Fare':>5}  {'Exp km':>6} {'Got km':>7} {'Dist':>5}  {'Conf':>6}")
print('-' * 95)
for r in rows:
    print(f"{r['Image']:<28} "
          f"{str(r['Exp Rs.']):>8} {str(r['Got Rs.']):>8} {r['Fare OK']:>5}  "
          f"{str(r['Exp km']):>6} {str(r['Got km']):>7} {r['Dist OK']:>5}  "
          f"{r['Conf']:>6.2f}")
print('=' * 95)

total    = len(rows)
fare_ok  = sum(1 for r in rows if r['Fare OK'] == 'PASS')
dist_ok  = sum(1 for r in rows if r['Dist OK'] == 'PASS')
avg_conf = sum(r['Conf'] for r in rows) / total if total else 0
print(f'\nFare accuracy: {fare_ok}/{total}  |  Distance accuracy: {dist_ok}/{total}  |  Avg confidence: {avg_conf:.2f}')

## Notes

- All logic is in `../ocr_engine.py` — edit that file, not this notebook.
- Run `python notebooks/ocr_runner.py` in the terminal for the same results without Jupyter.
- When you're satisfied with accuracy on your real screenshots, reply **Step 1 confirmed** and the FastAPI service (Step 2) will be built.